# Export from Sebastian Gomez's Git 
## Functions can be found in export_slsne_photometry.py

## Run the next two cells, source paths for repo, SNe directory of Sebastian's git, output location. 

In [96]:
# --- In your notebook ---
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import pandas as pd
from itertools import islice
from IPython.display import display
import re
from collections import Counter
import unicodedata



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [98]:
%reload_ext autoreload


In [100]:

repo_root = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# sanity checks
assert (repo_root / "py_files").is_dir(), "py_files/ folder not found"
assert (repo_root / "py_files" / "export_slsne_photometry.py").is_file(), "module file not found"

from py_files.export_slsne_photometry import (
    read_supernova_table_txt, to_export,
    process_one, process_all_events, load_allparams_robust, print_cols, resolve_cols 
)

supernovae_dir = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/supernovae")
out_perevent   = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files")
out_allevent   = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events")


## Parameter Control, Test directory locations, and presence of specific event can be found. 

In [103]:

# Output format controls
write_parquet = True   # set False if you don't want parquet
write_csv = True   # CSV mirror for auditing

# Quick sanity checks
print("Repo root:        ", repo_root)
print("Supernovae dir:   ", supernovae_dir, "exists:", supernovae_dir.exists())
print("Output directory: ", out_perevent, "exists:", out_perevent.exists())


# quick smoke test (replace with an event folder that exists in your clone)
test_event = "2005ap"
print((supernovae_dir / test_event / f"{test_event}.txt").exists())


Repo root:         /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric
Supernovae dir:    /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/supernovae exists: True
Output directory:  /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files exists: True
True


## Process an indivudal event .txt file into a table and see if it has been created. This will not create the csv. 

In [106]:

# IMPORTANT: pass the directory for the event, not the string's .parent
df_one = process_one(supernovae_dir / "2005ap", "2005ap",
                     out_perevent, write_parquet=True, write_csv=True)
display(df_one.head(10))

# Test on 2005ap


print("Rows:", len(df_one))
print("detected value counts:\n", df_one["detected"].value_counts(dropna=False))
print("CSV exists? ", (out_perevent / "2005ap.csv").exists())
print("Parquet exists? ", (out_perevent / "2005ap.parquet").exists())


,mjd,mag,mag_err,UL,filter,System,detected
0,53377.4,19.618,NaN,True,R,Vega,0
1,53379.4,18.938,NaN,True,R,Vega,0
2,53384.4,19.228,NaN,True,R,Vega,0
3,53386.5,19.438,NaN,True,R,Vega,0
4,53387.5,19.138,NaN,True,R,Vega,0
5,53389.5,18.948,NaN,True,R,Vega,0
6,53408.4,19.688,NaN,True,R,Vega,0
7,53415.4,18.648,NaN,True,R,Vega,0
8,53430.2,18.628,NaN,True,R,Vega,0
9,53432.3,18.498,0.11,False,R,Vega,1


Rows: 50
detected value counts:
 detected
1    32
0    18
Name: count, dtype: int64
CSV exists?  True
Parquet exists?  True


## Read original txt file in a table format, display it, Export the modified table to csv/parquet, display it

In [109]:

# Adjust this to your local path
file_2005ap = supernovae_dir / "2005ap" / "2005ap.txt"

raw = read_supernova_table_txt(file_2005ap)
print("Columns:", list(raw.columns))     # should include MJD, Mag, MagErr, Filter, UL, etc.
display(raw.head(5))

df = to_export(raw)
display(df.head(30))

# Quick checks
print("NaNs per column:\n", df.isna().sum())
print("detected counts:\n", df["detected"].value_counts(dropna=False))
print("unique filters:", df["filter"].unique())


Columns: ['MJD', 'Mag', 'Raw', 'MagErr', 'Telescope', 'Instrument', 'Filter', 'UL', 'System', 'Ignore', 'Source']


,MJD,Mag,Raw,MagErr,Telescope,Instrument,Filter,UL,System,Ignore,Source
0,53377.4,19.618,19.64,-1.0,ROTSE-III,--,R,True,Vega,True,2007ApJ...668L..99Q
1,53379.4,18.938,18.96,-1.0,ROTSE-III,--,R,True,Vega,True,2007ApJ...668L..99Q
2,53384.4,19.228,19.25,-1.0,ROTSE-III,--,R,True,Vega,True,2007ApJ...668L..99Q
3,53386.5,19.438,19.46,-1.0,ROTSE-III,--,R,True,Vega,True,2007ApJ...668L..99Q
4,53387.5,19.138,19.16,-1.0,ROTSE-III,--,R,True,Vega,True,2007ApJ...668L..99Q


,mjd,mag,mag_err,UL,filter,System,detected
0,53377.4000,19.6180,NaN,True,R,Vega,0
1,53379.4000,18.9380,NaN,True,R,Vega,0
2,53384.4000,19.2280,NaN,True,R,Vega,0
3,53386.5000,19.4380,NaN,True,R,Vega,0
4,53387.5000,19.1380,NaN,True,R,Vega,0
5,53389.5000,18.9480,NaN,True,R,Vega,0
6,53408.4000,19.6880,NaN,True,R,Vega,0
7,53415.4000,18.6480,NaN,True,R,Vega,0
8,53430.2000,18.6280,NaN,True,R,Vega,0
9,53432.3000,18.4980,0.1100,False,R,Vega,1


NaNs per column:
 mjd          0
mag          0
mag_err     18
UL           0
filter       0
System       0
detected     0
dtype: int64
detected counts:
 detected
1    32
0    18
Name: count, dtype: int64
unique filters: <StringArray>
['R', 'V', 'B', 'I']
Length: 4, dtype: string


## Process all events in /supernovae folder, put into csv and parquet for each event, put all events into one single csv and parquet 

In [112]:

index_df, combined_csv, combined_parq = process_all_events(
    supernovae_dir, out_perevent, out_allevent,
    include_ul=True, make_combined=True,
    write_parquet=True, write_csv=True
)
display(index_df.head(15))
print("Index:", out_perevent / "_index.csv")
print("Combined CSV:", combined_csv)
print("Combined Parquet:", combined_parq)

Exporting SLSNe per-object:   0%|          | 0/265 [00:00<?, ?it/s]

,event,status,path,n_rows,n_detected,n_limits
0,1991D,ok,/Users/andradenebula/Documents/Research/Transi...,19,17,2
1,1999as,ok,/Users/andradenebula/Documents/Research/Transi...,43,43,0
2,1999bz,ok,/Users/andradenebula/Documents/Research/Transi...,1,1,0
3,2002gh,ok,/Users/andradenebula/Documents/Research/Transi...,45,45,0
4,2005ap,ok,/Users/andradenebula/Documents/Research/Transi...,50,32,18
5,2006oz,ok,/Users/andradenebula/Documents/Research/Transi...,95,70,25
6,2007bi,ok,/Users/andradenebula/Documents/Research/Transi...,138,133,5
7,2009cb,ok,/Users/andradenebula/Documents/Research/Transi...,31,31,0
8,2009jh,ok,/Users/andradenebula/Documents/Research/Transi...,22,22,0
9,2010gx,ok,/Users/andradenebula/Documents/Research/Transi...,273,260,13


Index: /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files/_index.csv
Combined CSV: /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events/all_objects.csv
Combined Parquet: /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events/all_objects.parquet


# All Parameter Table 

##  Pull the all parameter table .txt, format it to see the first few lines, then print all the column names with indices

In [116]:

# Build the path (yours already does this)
allparams_path = repo_root / "SLSNe" / "slsne" / "ref_data" / "all_parameters.txt"
print("all_parameters path:", allparams_path)
assert allparams_path.exists(), "Could not find all_parameters.txt at the expected path."

# 1) Load the table
from py_files.export_slsne_photometry import load_allparams_robust
allparams_df = load_allparams_robust(allparams_path)



# 2) Print first few rows (optional)
from IPython.display import display
display(allparams_df.head())

# 3) Print column names with indices
for i, c in enumerate(allparams_df.columns):
    print(f"{i:>3}: {c}")


all_parameters path: /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/all_parameters.txt


,name,redshift_lo,redshift_med,redshift_up,texplosion_lo,texplosion_med,texplosion_up,fnickel_lo,fnickel_med,fnickel_up,...,r_peak_up,frac_lo,frac_med,frac_up,1frac_lo,1frac_med,1frac_up,efficiency_lo,efficiency_med,efficiency_up
0,1991D,0.0,0.04179,0.0,8.85557,-36.74645,6.08271,0.02723,0.03564,0.09394,...,0.36293,0.04075,0.99688,0.00303,0.00303,0.00312,0.04075,0.06205,0.11002,0.1464
1,1999as,0.0,0.127,0.0,9.61176,-32.81598,5.35865,0.00246,0.00389,0.00564,...,0.81024,0.00034,0.9999,8e-05,8e-05,0.0001,0.00034,0.03887,0.07776,0.08389
2,1999bz,0.0,0.0846,0.0,38.45281,-142.03278,50.37476,0.0918,0.336,0.11103,...,0.21678,0.03178,0.03205,0.49241,0.49241,0.96795,0.03178,0.38871,0.85337,0.6155
3,2002gh,0.0,0.3653,0.0,7.07357,-89.59422,9.65322,0.01792,0.02058,0.08967,...,0.07914,0.02044,0.99256,0.00642,0.00642,0.00744,0.02044,0.33218,0.91064,0.56285
4,2005ap,0.0,0.2832,0.0,4.27156,-8.92517,2.45517,0.01018,0.01341,0.04841,...,0.24393,0.01106,0.99808,0.00179,0.00179,0.00192,0.01106,0.1707,0.26925,0.32174


  0: name
  1: redshift_lo
  2: redshift_med
  3: redshift_up
  4: texplosion_lo
  5: texplosion_med
  6: texplosion_up
  7: fnickel_lo
  8: fnickel_med
  9: fnickel_up
 10: Pspin_lo
 11: Pspin_med
 12: Pspin_up
 13: log(Bfield)_lo
 14: log(Bfield)_med
 15: log(Bfield)_up
 16: Mns_lo
 17: Mns_med
 18: Mns_up
 19: thetaPB_lo
 20: thetaPB_med
 21: thetaPB_up
 22: mejecta_lo
 23: mejecta_med
 24: mejecta_up
 25: kappa_lo
 26: kappa_med
 27: kappa_up
 28: kappagamma_lo
 29: kappagamma_med
 30: kappagamma_up
 31: vejecta_lo
 32: vejecta_med
 33: vejecta_up
 34: temperature_lo
 35: temperature_med
 36: temperature_up
 37: alpha_lo
 38: alpha_med
 39: alpha_up
 40: cutoff_wavelength_lo
 41: cutoff_wavelength_med
 42: cutoff_wavelength_up
 43: log(nhhost)_lo
 44: log(nhhost)_med
 45: log(nhhost)_up
 46: A_V_lo
 47: A_V_med
 48: A_V_up
 49: MJD0_lo
 50: MJD0_med
 51: MJD0_up
 52: log(kenergy)_lo
 53: log(kenergy)_med
 54: log(kenergy)_up
 55: mnickel_lo
 56: mnickel_med
 57: mnickel_up
 58: log

##  Load all param table, preview first column, select columns you want to include besides the anchor, resolve indices to acutal names from list, build filename from selected columns, write to csv and parquet. Pull redshift or any other parameter from Sebastian's all parameter table and make it into a csv/parquet 

In [119]:

# --- load the table ---
allparams_df = load_allparams_robust(allparams_path)

# save all params.txt as a csv (complete)
if write_csv:
    allparams_df.to_csv(out_allevent / f"allparameter.csv", index=False)



In [121]:

## SAve parts of the all_params.txt table as a table 

# Preview just the anchor column (first col or 'name' if present)
print_cols(allparams_df, None, head=10)

# Choose columns by index (0-based). Example: pick column #2 besides the anchor.
selected_cols = [2]

# Resolve to actual names (prepend anchor)
resolved = resolve_cols(allparams_df, selected_cols)
df_cols = allparams_df[resolved].copy()

# Coerce numerics for non-anchor columns
for c in resolved[1:]:
    df_cols[c] = pd.to_numeric(df_cols[c], errors="coerce")

# Build filename stub from the *names* for your selected indices (excluding anchor)
idx_names = [allparams_df.columns[i] for i in selected_cols if 0 <= i < len(allparams_df.columns)]
sanitize = lambda s: re.sub(r"[^A-Za-z0-9._-]+", "_", s)
stub = "_".join(sanitize(c) for c in idx_names) if idx_names else "cols"

# Write outputs
if write_parquet:
    df_cols.to_parquet(out_allevent / f"allevent_{stub}.parquet",
                       index=False, engine="pyarrow", compression="zstd", compression_level=7)
if write_csv:
    df_cols.to_csv(out_allevent / f"allevent_{stub}.csv", index=False)

display(df_cols.head(10))
print("Columns used (resolved):", resolved)
print("Filename stub:", stub)

,name
0,1991D
1,1999as
2,1999bz
3,2002gh
4,2005ap
5,2006oz
6,2007bi
7,2009cb
8,2009jh
9,2010gx


,name,redshift_med
0,1991D,0.04179
1,1999as,0.12700
2,1999bz,0.08460
3,2002gh,0.36530
4,2005ap,0.28320
5,2006oz,0.37600
6,2007bi,0.12790
7,2009cb,0.18670
8,2009jh,0.34990
9,2010gx,0.22970


Columns used (resolved): ['name', 'redshift_med']
Filename stub: redshift_med


In [123]:

# read the file (adjust path as needed)
df = pd.read_csv(out_allevent / "all_objects.csv")

# count unique filters
filter_counts = df["filter"].value_counts()

# print results in the format you want
for f, count in filter_counts.items():
    print(f"{f}: {count} observations")


r: 15188 observations
g: 12410 observations
orange: 8971 observations
i: 5150 observations
cyan: 3037 observations
z: 1566 observations
B: 866 observations
V: 758 observations
R: 522 observations
I: 493 observations
G: 393 observations
U: 308 observations
UVM2: 291 observations
UVW1: 285 observations
UVW2: 284 observations
u: 255 observations
C: 170 observations
w: 165 observations
H: 163 observations
J: 162 observations
y: 90 observations
K: 76 observations
Ks: 53 observations
W1: 44 observations
W2: 44 observations
Y: 30 observations
F775W: 10 observations
F850LP: 9 observations
F625W: 7 observations
Rs: 3 observations
F475W: 2 observations
v: 2 observations


In [126]:

root = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files")

def clean_colname(c: str) -> str:
    # normalize unicode; strip BOM & whitespace; collapse internal spaces
    c = unicodedata.normalize("NFKC", str(c)).replace("\ufeff", "")
    c = re.sub(r"\s+", " ", c.strip())
    return c

totals = Counter()
skipped = []

for csv_path in sorted(root.glob("*.csv")):
    try:
        # peek header
        hdr = pd.read_csv(csv_path, nrows=0).columns
        cleaned = {col: clean_colname(col) for col in hdr}
        # find a column that cleans to exactly "filter"
        candidates = [orig for orig, cc in cleaned.items() if cc.lower() == "filter"]
        if not candidates:
            skipped.append((csv_path.name, list(hdr)))
            continue
        filter_col = candidates[0]

        # stream only that column
        for chunk in pd.read_csv(csv_path, usecols=[filter_col], dtype=str, chunksize=100_000):
            vals = chunk[filter_col].dropna().map(lambda s: s.strip())
            totals.update(vals)
    except Exception as e:
        skipped.append((csv_path.name, f"error: {e}"))

# Print results
for filt, count in totals.most_common():
    print(f"{filt}: {count} observations")

# Report any files we couldn’t parse a 'filter' column from (optional)
if skipped:
    print("\nSkipped files (no parsable 'filter' column or error):")
    for name, info in skipped:
        print(f"  {name} -> {info}")


r: 15188 observations
g: 12410 observations
orange: 8971 observations
i: 5150 observations
cyan: 3037 observations
z: 1566 observations
B: 866 observations
V: 758 observations
R: 522 observations
I: 493 observations
G: 393 observations
U: 308 observations
UVM2: 291 observations
UVW1: 285 observations
UVW2: 284 observations
u: 255 observations
C: 170 observations
w: 165 observations
H: 163 observations
J: 162 observations
y: 90 observations
K: 76 observations
Ks: 53 observations
W2: 44 observations
W1: 44 observations
Y: 30 observations
F775W: 10 observations
F850LP: 9 observations
F625W: 7 observations
Rs: 3 observations
F475W: 2 observations
v: 2 observations

Skipped files (no parsable 'filter' column or error):
  _index.csv -> ['event', 'status', 'path', 'n_rows', 'n_detected', 'n_limits']


In [128]:

folder = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/filters")

for path in folder.iterdir():
    if path.is_file():
        print(path.name)


Roman_WFI.F158.dat
2MASS_2MASS.Ks_AB.dat
Roman_WFI.F213.dat
PAN-STARRS_PS1.z_AB.dat
Generic_Bessell.B_AB.dat
Generic_Bessell.V_AB.dat
Palomar_ZTF.r_AB.dat
SLOAN_SDSS.r_AB.dat
WISE_WISE.W2_AB.dat
Swift_UVOT.U_AB.dat
Roman_WFI.F106.dat
SLOAN_SDSS.z_AB.dat
Roman_WFI.F129.dat
PAN-STARRS_PS1.r_AB.dat
Generic_Bessell.R_AB.dat
Swift_UVOT.UVW2_AB.dat
Roman_WFI.F062.dat
Palomar_ZTF.g_AB.dat
Swift_UVOT.V_AB.dat
WISE_WISE.W1_AB.dat
2MASS_2MASS.J_AB.dat
SLOAN_SDSS.i_AB.dat
Roman_WFI.F087.dat
Swift_UVOT.B_AB.dat
PAN-STARRS_PS1.y_AB.dat
2MASS_2MASS.H_AB.dat
SLOAN_SDSS.g_AB.dat
Generic_Bessell.U_AB.dat
Palomar_ZTF.i_AB.dat
PAN-STARRS_PS1.i_AB.dat
Swift_UVOT.UVM2_AB.dat
Roman_WFI.F146.dat
Generic_Bessell.I_AB.dat
Roman_WFI.F184.dat
Swift_UVOT.UVW1_AB.dat
PAN-STARRS_PS1.g_AB.dat
SLOAN_SDSS.u_AB.dat


In [141]:
from pathlib import Path
import pandas as pd

# input files
REF_DIR = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data")
FILTER_REF_TXT   = REF_DIR / "filter_reference.txt"
GENERIC_REF_TXT  = REF_DIR / "generic_reference.txt"

# output dir
OUT_DIR = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILTER_REF_CSV   = OUT_DIR / "filter_reference.csv"
GENERIC_REF_CSV  = OUT_DIR / "generic_reference.csv"

def convert_txt_to_csv(txt_path: Path, csv_path: Path):
    """
    Convert space/tab-delimited reference text file into CSV.
    Preserves any non-numeric content (e.g., 'Generic') in all columns.
    """
    # Read: tolerate multiple spaces / tabs, ignore comment lines
    df = pd.read_csv(txt_path, sep=r"\s{2,}|\t", engine="python", comment="#", dtype=str)
    # Clean headers and strip whitespace in all string fields
    df.columns = [c.strip() for c in df.columns]
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()

    # Write as-is; do NOT coerce/blank 'Generic' or other strings
    df.to_csv(csv_path, index=False)
    print(f"[INFO] Saved {csv_path} with {len(df)} rows and columns {list(df.columns)}")

if __name__ == "__main__":
    convert_txt_to_csv(FILTER_REF_TXT, FILTER_REF_CSV)
    convert_txt_to_csv(GENERIC_REF_TXT, GENERIC_REF_CSV)


[INFO] Saved /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events/filter_reference.csv with 539 rows and columns ['Filter', 'Telescope', 'Instrument', 'System', 'Cenwave', 'Zeropoint']
[INFO] Saved /Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events/generic_reference.csv with 41 rows and columns ['Filter', 'System', 'Cenwave', 'Zeropoint']


In [148]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import OrderedDict

# ---------- Paths ----------
PER_EVENT_DIR = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files")
REF_OUT_DIR   = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events")
FILTER_REF_CSV   = REF_OUT_DIR / "filter_reference.csv"
GENERIC_REF_CSV  = REF_OUT_DIR / "generic_reference.csv"

# ---------- Helpers ----------
def _is_numeric(x):
    try:
        return np.isfinite(float(x))
    except Exception:
        return False

def _load_ref_table(csv_path: Path) -> pd.DataFrame:
    """
    Load reference CSV as strings, strip whitespace.
    Expects columns: Filter, System, Cenwave
    """
    df = pd.read_csv(csv_path, dtype=str).fillna("")
    df.columns = [c.strip() for c in df.columns]
    need = {"Filter", "System", "Cenwave"}
    miss = need - set(df.columns)
    if miss:
        raise RuntimeError(f"{csv_path} missing columns: {miss}")
    for c in need:
        df[c] = df[c].astype(str).str.strip()
    return df[list(need)]

def _collapse_numeric_cenwave(df: pd.DataFrame):
    """
    Group by (Filter,System) and take the MEDIAN of numeric Cenwave entries.
    Returns dict {(Filter, System): Cenwave_float}.
    """
    df_num = df[df["Cenwave"].apply(_is_numeric)].copy()
    if df_num.empty:
        return {}
    df_num["Cenwave_float"] = df_num["Cenwave"].astype(float)
    med = df_num.groupby(["Filter","System"], as_index=False)["Cenwave_float"].median()
    return {(r["Filter"], r["System"]): float(r["Cenwave_float"]) for _, r in med.iterrows()}

def _lookup_cenwave(filter_label: str, system: str,
                    map_filterref: dict, df_filterref: pd.DataFrame,
                    map_generic: dict) -> float:
    """
    Priority:
      1) filter_reference.csv exact (Filter,System) if Cenwave is numeric (not 'Generic').
      2) generic_reference.csv exact (Filter,System) if numeric.
      3) generic_reference.csv system-agnostic (prefer AB, then Vega, then blank).
    Returns float wavelength in Å, or np.nan if not found.
    """
    flab = filter_label.strip()
    sys  = system.strip()

    # 1) numeric entry in filter_reference?
    if (flab, sys) in map_filterref:
        return map_filterref[(flab, sys)]

    # 2) generic_reference exact
    if (flab, sys) in map_generic:
        return map_generic[(flab, sys)]

    # 3) generic_reference system-agnostic (AB, Vega, then blank)
    for s in ("AB", "Vega", ""):
        if (flab, s) in map_generic:
            return map_generic[(flab, s)]

    return np.nan

# ---------- Build the map ----------
def build_user_filter_map(
    per_event_dir: Path = PER_EVENT_DIR,
    filter_reference_csv: Path = FILTER_REF_CSV,
    generic_reference_csv: Path = GENERIC_REF_CSV,
    verbose: bool = True,
):
    # Discover distinct (Filter,System) used in per-event CSVs
    pairs = set()
    for csv in sorted(per_event_dir.glob("*.csv")):
        try:
            df = pd.read_csv(csv, dtype=str).fillna("")
        except Exception:
            continue
        # normalize headers
        cols = {c.strip().lower(): c for c in df.columns}
        if "filter" not in cols:
            continue
        filt_col = cols["filter"]
        # typical header is 'System'; if absent, treat as blank
        sys_col  = cols.get("system", None)

        if sys_col:
            for f, s in df[[filt_col, sys_col]].itertuples(index=False):
                pairs.add((str(f).strip(), str(s).strip()))
        else:
            for f in df[filt_col].astype(str):
                pairs.add((str(f).strip(), ""))

    # Load references
    dfr = _load_ref_table(filter_reference_csv)
    dfg = _load_ref_table(generic_reference_csv)

    # Build (Filter,System) -> Cenwave maps
    map_filterref = _collapse_numeric_cenwave(dfr)  # ONLY numeric from filter_reference
    map_generic   = _collapse_numeric_cenwave(dfg)  # numeric from generic_reference

    # Build user_filter_map keyed by LOWER-CASED filter
    user_filter_map = OrderedDict()
    unresolved = []

    for flab, sys in sorted(pairs):
        lam = _lookup_cenwave(flab, sys, map_filterref, dfr, map_generic)
        if np.isfinite(lam) and lam > 0:
            user_filter_map[flab.lower()] = float(lam)
        else:
            unresolved.append((flab, sys))

    if verbose:
        print(f"[discover] {len(pairs)} distinct (Filter,System) pairs.")
        print(f"[resolve]  {len(user_filter_map)} mapped, {len(unresolved)} unresolved.")
        if unresolved:
            print("  Unresolved examples (up to 10):", unresolved[:10])

    return user_filter_map

# ---------- Run & pretty-print ----------
if __name__ == "__main__":
    ufm = build_user_filter_map()
    print("\nuser_filter_map = {")
    for k, v in ufm.items():
        print(f'    "{k}": {v:.2f},')
    print("}")


[discover] 51 distinct (Filter,System) pairs.
[resolve]  26 mapped, 0 unresolved.

user_filter_map = {
    "b": 4332.70,
    "c": 5627.80,
    "f475w": 4708.87,
    "f625w": 6266.20,
    "f775w": 7652.44,
    "f850lp": 9004.99,
    "g": 4671.78,
    "h": 16230.17,
    "i": 7682.36,
    "j": 12317.97,
    "k": 21682.12,
    "ks": 21454.68,
    "r": 6141.12,
    "rs": 6734.34,
    "u": 3608.04,
    "uvm2": 2245.03,
    "uvw1": 2681.67,
    "uvw2": 2083.95,
    "v": 3878.68,
    "w1": 33526.00,
    "w2": 46028.00,
    "y": 9613.60,
    "cyan": 5182.42,
    "orange": 6629.82,
    "w": 5980.70,
    "z": 8906.54,
}


In [192]:

# --- Configure your root directory here ---
ROOT = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/supernovae")

def _read_filter_column(txt_path: Path) -> list[str]:
    """
    Read values from the 'Filter' column (case-insensitive header match).
    Returns [] if the file doesn't exist, isn't parseable, or lacks the column.
    """
    if not txt_path.exists():
        return []
    try:
        df = pd.read_csv(
            txt_path,
            sep=r"\s+",
            engine="python",
            comment="#",
            dtype=str,
            na_filter=False
        )
        col = next((c for c in df.columns if c.lower() == "filter"), None)
        if col is None:
            return []
        vals = [v for v in df[col].tolist() if isinstance(v, str) and v.strip() != ""]
        return vals
    except Exception:
        return []

def collect_unique_filters(root: Path) -> list[str]:
    """
    Walk subfolders and collect filters from:
      (folder).txt, (folder)_model.txt, (folder)_rest.txt
    Returns a case-insensitive sorted list (values unmodified).
    """
    seen = set()
    for folder in sorted([p for p in root.iterdir() if p.is_dir()]):
        base = folder.name
        candidates = [
            folder / f"{base}.txt",
            folder / f"{base}_model.txt",
            folder / f"{base}_rest.txt",
        ]
        for p in candidates:
            for v in _read_filter_column(p):
                seen.add(v)  # store exactly as found, no normalization
    # sort case-insensitively while preserving original casing in output
    return sorted(seen, key=lambda s: (s.lower(), s))

unique_filters = collect_unique_filters(ROOT)

print("Unique filters found (exact values):")
for f in unique_filters:
    print(f)

print(f"\nTotal unique filters: {len(unique_filters)}")


Unique filters found (exact values):
B
B-AB
B-CPCS
B-FLWO
B-Helmos
B-LCO
B-LCOGT
B-Lulin
B-MDM
B-P60
B-SLT
B-Swift
B-Vega
C
cyan
F475W
F625W
F775W
F850LP
G
g
g-ASASSN
g-Blanco
g-FLWO
g-FTN
g-GN
g-Keck
g-LCO
g-LCOGT
g-LT
g-Magellan
g-NTT
g-P48
g-P60
g-PS1
g-SkyMapper
g-Subaru
H
H-TNG
I
i
i---
i-Blanco
i-FLWO
i-GN
i-GROND
i-LCO
i-LCOGT
i-LT
i-Magellan
i-NTT
i-P200
i-P48
i-P60
i-PS1
i-SkyMapper
i-Subaru
J
K
K-TNG
Ks
orange
R
r
r---
r-Blanco
r-FLWO
r-GN
r-GROND
r-LCO
r-LCOGT
r-LT
r-Magellan
r-MMT
r-NTT
r-P200
r-P48
r-P60
r-PS1
r-SkyMapper
r-Subaru
Rs
swift_B
swift_U
swift_UVM2
swift_UVW1
swift_UVW2
swift_V
U
u
U-AB
U-LCO
U-LCOGT
u-LT
u-NTT
u-P60
U-Sampurnanand
U-Swift
U-Vega
UVM2
UVM2-AB
UVM2-Vega
UVW1
UVW1-AB
UVW1-Vega
UVW2
UVW2-AB
UVW2-Vega
V
v
V-CPCS
V-FLWO
V-LCO
V-LCOGT
V-LSQ
V-MDM
V-NOT
V-Sampurnanand
V-SLT
V-Swift
w
W1
W2
Y
y
z
z-Blanco
z-GN
z-GROND
z-LCO
z-LT
z-Magellan
z-NTT
z-PS1
z-Subaru

Total unique filters: 135
